### **Transformer encoder textual**


Este cuaderno repasa los conceptos textuales que un estudiante necesita dominar antes de trabajar con Transformers multimodales. El objetivo no es repetir todo el Transformer desde cero, sino consolidar las piezas que luego reaparecen en VisualBERT, ViLBERT, LXMERT, UNITER, ViLT, BLIP-2, LLaVA, Qwen2.5-VL e InternVL.

El cuaderno se concentra en BERT como modelo encoder-only para lenguaje, atención bidireccional, masked language modeling, embeddings de token, posición y segmento, freezing, fine-tuning y el puente conceptual hacia tokens visuales y proyectores multimodales.


### **Objetivos de aprendizaje**


Al finalizar el cuaderno, el estudiante debe poder:

1. Explicar por qué BERT es un Transformer encoder-only y por qué eso importa para modelos multimodales de comprensión.
2. Diferenciar atención bidireccional de atención causal.
3. Construir entradas tipo BERT con `[CLS]`, `[SEP]`, `[MASK]`, `[PAD]`, ids de segmento y máscara de atención.
4. Implementar un masked language modeling simplificado con PyTorch.
5. Interpretar freezing como estrategia de transferencia y reducción de costo.
6. Comparar freezing, fine-tuning completo y fine-tuning parcial.
7. Entender cómo los embeddings textuales de BERT se conectan con embeddings visuales en modelos como VisualBERT, UNITER, ViLT, BLIP-2 y LLaVA.

#### **Conexión con Transformers multimodales**

En modelos multimodales, muchas decisiones arquitectónicas vienen de BERT:

| Concepto textual | Papel en BERT | Papel en modelos multimodales |
|---|---|---|
| Token embedding | Representa palabras o subpalabras | Se combina con regiones, patches o tokens visuales |
| Position embedding | Indica orden dentro de la secuencia | Indica orden textual o posición de tokens visuales |
| Segment embedding | Distingue frases A y B | Puede distinguir texto, imagen, pregunta, respuesta o región |
| `[CLS]` | Resume la secuencia para clasificación | Sirve como token global para tareas multimodales |
| `[MASK]` | Permite preentrenamiento MLM | Inspira objetivos como MRM y masking multimodal |
| Atención bidireccional | Cada token atiende a todos los tokens visibles | Permite fusión profunda entre texto e imagen |
| Freezing | Reduce costo de entrenamiento | Se usa en BLIP-2, Frozen, LLaVA y modelos modernos |

### **Configuración inicial**

#### **Importación de bibliotecas**

El cuaderno usa PyTorch, pandas, numpy y matplotlib. No requiere descargar modelos externos. Las celdas opcionales con Hugging Face están desactivadas por defecto para mantener reproducibilidad en CPU y en Docker.

In [1]:
# Importaciones principales del cuaderno
import math
import random
import re
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Semillas para hacer reproducibles los resultados del laboratorio
SEMILLA = 225
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

# Selección de dispositivo
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo disponible:", dispositivo)
print("Versión de PyTorch:", torch.__version__)

Dispositivo disponible: cpu
Versión de PyTorch: 2.4.1+cpu


### **Del Transformer textual a BERT**

#### **BERT como encoder-only**

BERT usa solo la parte encoder del Transformer. Esto significa que procesa una secuencia completa con atención bidireccional. A diferencia de un decoder-only, no está limitado a mirar solo tokens anteriores.

Esta diferencia es crucial:

| Modelo | Tipo de atención | Uso típico |
|---|---|---|
| BERT | bidireccional | comprensión, clasificación, extracción, VQA como encoder |
| GPT | causal | generación autoregresiva |
| Encoder-decoder | bidireccional en encoder y causal en decoder | traducción, captioning, generación condicionada |

En multimodalidad, muchos modelos de comprensión siguen la lógica encoder-only de BERT. VisualBERT, UNITER y ViLT procesan tokens textuales y visuales en una representación fusionada.

### **Tokens especiales y vocabulario mínimo**

#### **Estructura de entrada tipo BERT**

BERT no recibe texto sin procesar. Recibe ids de tokens. En este cuaderno usamos un tokenizador simple para que el mecanismo sea transparente.

Los tokens especiales principales son:

| Token | Función |
|---|---|
| `[PAD]` | Relleno para igualar longitudes |
| `[UNK]` | Token desconocido |
| `[CLS]` | Representación global de la secuencia |
| `[SEP]` | Separador de segmentos |
| `[MASK]` | Token usado para masked language modeling |

In [2]:
# Corpus pequeño en español para un laboratorio textual
corpus = [
    "bert aprende representaciones contextuales del lenguaje",
    "el transformer encoder usa atencion bidireccional",
    "masked language modeling oculta tokens durante el preentrenamiento",
    "los embeddings de posicion conservan informacion del orden",
    "los embeddings de segmento separan pares de oraciones",
    "el token cls resume la secuencia completa",
    "el token sep separa dos segmentos de texto",
    "freezing permite entrenar pocas capas del modelo",
    "fine tuning completo actualiza todos los parametros",
    "la atencion calcula relaciones entre tokens",
    "un encoder textual puede recibir embeddings visuales proyectados",
    "visualbert fusiona regiones visuales y tokens textuales",
    "uniter aprende alineamiento entre palabras y regiones",
    "vilt procesa patches visuales con texto en una sola pila",
    "blip dos usa un q former entre vision y lenguaje",
    "llava conecta un encoder visual con un modelo de lenguaje",
    "qwen vl trabaja con documentos diagramas y grounding",
    "internvl estudia escalamiento razonamiento y evaluacion multimodal",
    "los modelos multimodales combinan comprension y generacion",
    "el grounding reduce respuestas sin evidencia visual"
]

tokens_especiales = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

def tokenizar(texto):
    """Tokeniza de forma simple usando minúsculas y separadores alfabéticos."""
    return re.findall(r"[a-záéíóúñ0-9]+", texto.lower())

conteo = Counter()
for oracion in corpus:
    conteo.update(tokenizar(oracion))

vocabulario = tokens_especiales + sorted(conteo.keys())
token_a_id = {tok: idx for idx, tok in enumerate(vocabulario)}
id_a_token = {idx: tok for tok, idx in token_a_id.items()}

print("Tamaño del vocabulario:", len(vocabulario))
print("Primeros tokens:", vocabulario[:20])

Tamaño del vocabulario: 111
Primeros tokens: ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', 'actualiza', 'alineamiento', 'aprende', 'atencion', 'bert', 'bidireccional', 'blip', 'calcula', 'capas', 'cls', 'combinan', 'completa', 'completo', 'comprension', 'con']


In [3]:
import re
import torch
import pandas as pd

MAX_LEN = 18

TOKENS_ESPECIALES = {"[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"}

PAD_ID = token_a_id["[PAD]"]
UNK_ID = token_a_id["[UNK]"]
CLS_ID = token_a_id["[CLS]"]
SEP_ID = token_a_id["[SEP]"]
MASK_ID = token_a_id["[MASK]"]


def tokenizar_basico(texto):
    """
    Tokenizador simple para el cuaderno.
    Conserva tokens especiales como [MASK] y normaliza a minusculas
    solamente las palabras comunes.
    """
    piezas = re.findall(r"\[[A-Z]+\]|[\wáéíóúñüÁÉÍÓÚÑÜ]+|[^\s\w]", texto)

    tokens = []
    for pieza in piezas:
        if pieza in TOKENS_ESPECIALES:
            tokens.append(pieza)
        else:
            tokens.append(pieza.lower())

    return tokens


def codificar_bert(texto_a, texto_b=None, max_len=MAX_LEN):
    """
    Codifica una o dos oraciones con formato tipo BERT.

    Caso de una oración:
    [CLS] texto_a [SEP]

    Caso de dos oraciones:
    [CLS] texto_a [SEP] texto_b [SEP]

    Devuelve:
    input_ids, token_type_ids, attention_mask y tokens.
    """
    tokens_a = tokenizar_basico(texto_a)

    pares_token_segmento = [("[CLS]", 0)]

    for token in tokens_a:
        pares_token_segmento.append((token, 0))

    pares_token_segmento.append(("[SEP]", 0))

    if texto_b is not None:
        tokens_b = tokenizar_basico(texto_b)

        for token in tokens_b:
            pares_token_segmento.append((token, 1))

        pares_token_segmento.append(("[SEP]", 1))

    if len(pares_token_segmento) > max_len:
        pares_token_segmento = pares_token_segmento[:max_len]
        ultimo_segmento = pares_token_segmento[-1][1]
        pares_token_segmento[-1] = ("[SEP]", ultimo_segmento)

    tokens = [token for token, _ in pares_token_segmento]
    token_type_ids = [segmento for _, segmento in pares_token_segmento]

    input_ids = [token_a_id.get(token, UNK_ID) for token in tokens]
    attention_mask = [1] * len(input_ids)

    while len(input_ids) < max_len:
        input_ids.append(PAD_ID)
        token_type_ids.append(0)
        attention_mask.append(0)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "tokens": tokens,
    }

ejemplo = codificar_bert(
    "el transformer encoder usa atencion bidireccional",
    "bert aprende representaciones contextuales",
    max_len=18,
)

tokens_con_padding = ejemplo["tokens"] + ["[PAD]"] * (18 - len(ejemplo["tokens"]))

pd.DataFrame({
    "posicion": list(range(18)),
    "token": tokens_con_padding,
    "input_id": ejemplo["input_ids"].tolist(),
    "segmento": ejemplo["token_type_ids"].tolist(),
    "mascara": ejemplo["attention_mask"].tolist(),
})

,posicion,token,input_id,segmento,mascara
0,0,[CLS],2,0,1
1,1,el,29,0,1
2,2,transformer,98,0,1
3,3,encoder,32,0,1
4,4,usa,103,0,1
5,5,atencion,8,0,1
6,6,bidireccional,10,0,1
7,7,[SEP],3,0,1
8,8,bert,9,1,1
9,9,aprende,7,1,1


### **Embeddings de token, posición y segmento**

#### **Suma de representaciones**

BERT combina tres fuentes de información:

1. Embedding de token.
2. Embedding de posición.
3. Embedding de segmento.

Esta suma permite que el encoder sepa qué símbolo aparece, dónde aparece y a qué segmento pertenece. En multimodalidad, esta idea se extiende con embeddings de modalidad, por ejemplo texto frente a imagen.

In [4]:
class EmbeddingsBERTMini(nn.Module):
    """Embeddings de token, posición y segmento en estilo BERT."""

    def __init__(self, vocab_size, d_model=64, max_len=64, num_segmentos=2, dropout=0.1):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, d_model)
        self.position_embeddings = nn.Embedding(max_len, d_model)
        self.segment_embeddings = nn.Embedding(num_segmentos, d_model)
        self.norma = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, token_type_ids):
        batch_size, seq_len = input_ids.shape
        posiciones = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, seq_len)

        x = (
            self.token_embeddings(input_ids)
            + self.position_embeddings(posiciones)
            + self.segment_embeddings(token_type_ids)
        )
        x = self.norma(x)
        x = self.dropout(x)
        return x

emb = EmbeddingsBERTMini(len(vocabulario), d_model=32, max_len=18)
input_ids = ejemplo["input_ids"].unsqueeze(0)
token_type_ids = ejemplo["token_type_ids"].unsqueeze(0)

with torch.no_grad():
    salida_emb = emb(input_ids, token_type_ids)

print("Forma de input_ids:", tuple(input_ids.shape))
print("Forma de embeddings:", tuple(salida_emb.shape))

Forma de input_ids: (1, 18)
Forma de embeddings: (1, 18, 32)


### **Atención bidireccional frente a atención causal**

#### **Qué cambia en la máscara**

En BERT, cada token puede atender a tokens anteriores y posteriores, excepto posiciones de padding. En un decoder autoregresivo, cada token solo puede atender a posiciones anteriores o a sí mismo.

Esta diferencia explica por qué BERT es fuerte para comprensión y por qué los LLMs decoder-only son fuertes para generación.

In [5]:
def matriz_mascara_bidireccional(tokens_visibles):
    """Crea una matriz donde todos los tokens visibles pueden atenderse entre sí."""
    mascara = np.ones((tokens_visibles, tokens_visibles), dtype=int)
    return mascara

def matriz_mascara_causal(tokens_visibles):
    """Crea una matriz causal donde cada token atiende solo a posiciones previas."""
    mascara = np.tril(np.ones((tokens_visibles, tokens_visibles), dtype=int))
    return mascara

tokens_visibles = int(ejemplo["attention_mask"].sum().item())
tokens_mostrados = ejemplo["tokens"]

bidireccional = matriz_mascara_bidireccional(tokens_visibles)
causal = matriz_mascara_causal(tokens_visibles)

print("Tokens visibles:", tokens_mostrados)
print("\nMáscara bidireccional:")
display(pd.DataFrame(bidireccional, index=tokens_mostrados, columns=tokens_mostrados))
print("\nMáscara causal:")
display(pd.DataFrame(causal, index=tokens_mostrados, columns=tokens_mostrados))

Tokens visibles: ['[CLS]', 'el', 'transformer', 'encoder', 'usa', 'atencion', 'bidireccional', '[SEP]', 'bert', 'aprende', 'representaciones', 'contextuales', '[SEP]']

Máscara bidireccional:


,[CLS],el,transformer,encoder,usa,atencion,bidireccional,[SEP],bert,aprende,representaciones,contextuales,[SEP]
[CLS],1,1,1,1,1,1,1,1,1,1,1,1,1
el,1,1,1,1,1,1,1,1,1,1,1,1,1
transformer,1,1,1,1,1,1,1,1,1,1,1,1,1
encoder,1,1,1,1,1,1,1,1,1,1,1,1,1
usa,1,1,1,1,1,1,1,1,1,1,1,1,1
atencion,1,1,1,1,1,1,1,1,1,1,1,1,1
bidireccional,1,1,1,1,1,1,1,1,1,1,1,1,1
[SEP],1,1,1,1,1,1,1,1,1,1,1,1,1
bert,1,1,1,1,1,1,1,1,1,1,1,1,1
aprende,1,1,1,1,1,1,1,1,1,1,1,1,1



Máscara causal:


,[CLS],el,transformer,encoder,usa,atencion,bidireccional,[SEP],bert,aprende,representaciones,contextuales,[SEP]
[CLS],1,0,0,0,0,0,0,0,0,0,0,0,0
el,1,1,0,0,0,0,0,0,0,0,0,0,0
transformer,1,1,1,0,0,0,0,0,0,0,0,0,0
encoder,1,1,1,1,0,0,0,0,0,0,0,0,0
usa,1,1,1,1,1,0,0,0,0,0,0,0,0
atencion,1,1,1,1,1,1,0,0,0,0,0,0,0
bidireccional,1,1,1,1,1,1,1,0,0,0,0,0,0
[SEP],1,1,1,1,1,1,1,1,0,0,0,0,0
bert,1,1,1,1,1,1,1,1,1,0,0,0,0
aprende,1,1,1,1,1,1,1,1,1,1,0,0,0


In [ ]:
# Visualización simple de las máscaras de atención
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(bidireccional)
axes[0].set_title("Atención bidireccional")
axes[0].set_xlabel("Token atendido")
axes[0].set_ylabel("Token que consulta")

axes[1].imshow(causal)
axes[1].set_title("Atención causal")
axes[1].set_xlabel("Token atendido")
axes[1].set_ylabel("Token que consulta")

plt.tight_layout()
plt.show()

### **Bloque encoder en estilo BERT**

#### **Componentes principales**

Un bloque encoder usa:

1. Multi-head self-attention.
2. Conexión residual.
3. LayerNorm.
4. Feed-forward network.
5. Otra conexión residual.
6. Otra LayerNorm.

Este patrón aparece también en modelos multimodales. La diferencia es que la secuencia puede contener tokens de texto, regiones visuales, patches o queries aprendibles.

In [ ]:
class BloqueEncoderBERTMini(nn.Module):
    """Bloque encoder con self-attention bidireccional."""

    def __init__(self, d_model=64, num_heads=4, d_ff=128, dropout=0.1):
        super().__init__()
        self.atencion = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norma_1 = nn.LayerNorm(d_model)
        self.norma_2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None, devolver_atencion=False):
        # La máscara de padding evita atender a posiciones de relleno.
        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = attention_mask == 0

        salida_atencion, pesos = self.atencion(
            x,
            x,
            x,
            key_padding_mask=key_padding_mask,
            need_weights=devolver_atencion,
            average_attn_weights=False,
        )

        x = self.norma_1(x + self.dropout(salida_atencion))
        salida_ffn = self.ffn(x)
        x = self.norma_2(x + self.dropout(salida_ffn))

        if devolver_atencion:
            return x, pesos
        return x

bloque = BloqueEncoderBERTMini(d_model=32, num_heads=4, d_ff=64)
with torch.no_grad():
    salida_bloque, pesos_atencion = bloque(
        salida_emb,
        ejemplo["attention_mask"].unsqueeze(0),
        devolver_atencion=True,
    )

print("Salida del bloque:", tuple(salida_bloque.shape))
print("Pesos de atención:", tuple(pesos_atencion.shape))

### **Masked language modeling**

#### **Idea central**

Masked language modeling (modelos de lenguaje enmascarados) consiste en ocultar algunos tokens y entrenar el modelo para reconstruirlos usando contexto izquierdo y derecho. Esto hace que el encoder aprenda representaciones bidireccionales.

La tarea no es generar una oración completa de izquierda a derecha. La tarea es inferir tokens faltantes a partir de una secuencia parcialmente observada.

In [ ]:
def aplicar_mlm(input_ids, prob_mascara=0.15):
    """Aplica una corrupción tipo MLM a una secuencia de ids."""
    input_ids = input_ids.clone()
    labels = torch.full_like(input_ids, fill_value=-100)

    tokens_no_mascarables = {PAD_ID, CLS_ID, SEP_ID}
    candidatos = [
        i for i, token_id in enumerate(input_ids.tolist())
        if token_id not in tokens_no_mascarables
    ]

    if len(candidatos) == 0:
        return input_ids, labels

    num_mascaras = max(1, int(round(len(candidatos) * prob_mascara)))
    posiciones = random.sample(candidatos, k=min(num_mascaras, len(candidatos)))

    for pos in posiciones:
        token_original = input_ids[pos].item()
        labels[pos] = token_original

        r = random.random()
        if r < 0.80:
            input_ids[pos] = MASK_ID
        elif r < 0.90:
            input_ids[pos] = random.randint(len(tokens_especiales), len(vocabulario) - 1)
        else:
            input_ids[pos] = token_original

    return input_ids, labels

ejemplo_mlm = codificar_bert("masked language modeling oculta tokens durante el preentrenamiento", max_len=14)
ids_corruptos, labels = aplicar_mlm(ejemplo_mlm["input_ids"], prob_mascara=0.30)

tabla_mlm = pd.DataFrame({
    "posicion": list(range(14)),
    "token_original": [id_a_token[i] for i in ejemplo_mlm["input_ids"].tolist()],
    "token_entrada": [id_a_token[i] for i in ids_corruptos.tolist()],
    "label_mlm": [id_a_token[i] if i != -100 else "ignorar" for i in labels.tolist()],
})
tabla_mlm

### **Modelo BERT mini desde cero**

El siguiente modelo no busca competir con BERT real. Su función es pedagógica:

1. Recibe ids de tokens, ids de segmento y máscara de atención.
2. Construye embeddings tipo BERT.
3. Procesa la secuencia con varios bloques encoder.
4. Usa una cabecera MLM para predecir tokens ocultos.
5. Expone la representación `[CLS]` para tareas de clasificación.

Esta arquitectura es suficiente para entender freezing y para conectar luego con modelos multimodales.

In [ ]:
class BERTMini(nn.Module):
    """BERT pequeño para masked language modeling y clasificación."""

    def __init__(
        self,
        vocab_size,
        d_model=64,
        max_len=32,
        num_segmentos=2,
        num_layers=2,
        num_heads=4,
        d_ff=128,
        dropout=0.1,
    ):
        super().__init__()
        self.embeddings = EmbeddingsBERTMini(
            vocab_size=vocab_size,
            d_model=d_model,
            max_len=max_len,
            num_segmentos=num_segmentos,
            dropout=dropout,
        )
        self.capas = nn.ModuleList([
            BloqueEncoderBERTMini(
                d_model=d_model,
                num_heads=num_heads,
                d_ff=d_ff,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])
        self.mlm_transform = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )
        self.mlm_decoder = nn.Linear(d_model, vocab_size, bias=False)
        self.bias_mlm = nn.Parameter(torch.zeros(vocab_size))

        # Weight tying entre embedding de tokens y decodificador MLM.
        self.mlm_decoder.weight = self.embeddings.token_embeddings.weight

    def forward(self, input_ids, token_type_ids, attention_mask, devolver_atencion=False):
        x = self.embeddings(input_ids, token_type_ids)
        atenciones = []

        for capa in self.capas:
            if devolver_atencion:
                x, pesos = capa(x, attention_mask=attention_mask, devolver_atencion=True)
                atenciones.append(pesos)
            else:
                x = capa(x, attention_mask=attention_mask, devolver_atencion=False)

        logits_mlm = self.mlm_decoder(self.mlm_transform(x)) + self.bias_mlm
        salida_cls = x[:, 0, :]

        if devolver_atencion:
            return logits_mlm, salida_cls, atenciones
        return logits_mlm, salida_cls

MAX_LEN = 18
modelo = BERTMini(
    vocab_size=len(vocabulario),
    d_model=64,
    max_len=MAX_LEN,
    num_layers=2,
    num_heads=4,
    d_ff=128,
).to(dispositivo)

num_parametros = sum(p.numel() for p in modelo.parameters())
print("Parámetros totales del modelo BERT mini:", num_parametros)

### **Preparación de datos para MLM**

#### **Lotes de entrenamiento**

Usaremos el corpus pequeño del cuaderno y repetiremos oraciones para construir lotes. El objetivo no es lograr un modelo de lenguaje robusto, sino observar el mecanismo de aprendizaje de MLM.

In [ ]:
def construir_dataset_mlm(corpus, repeticiones=20, max_len=MAX_LEN):
    """Construye ejemplos corruptos para entrenamiento MLM."""
    ejemplos = []
    for _ in range(repeticiones):
        for texto in corpus:
            codificado = codificar_bert(texto, max_len=max_len)
            entrada_mlm, labels = aplicar_mlm(codificado["input_ids"], prob_mascara=0.20)
            ejemplos.append({
                "input_ids": entrada_mlm,
                "token_type_ids": codificado["token_type_ids"],
                "attention_mask": codificado["attention_mask"],
                "labels": labels,
            })
    random.shuffle(ejemplos)
    return ejemplos

def crear_lote(ejemplos):
    """Agrupa ejemplos individuales en tensores."""
    return {
        "input_ids": torch.stack([e["input_ids"] for e in ejemplos]).to(dispositivo),
        "token_type_ids": torch.stack([e["token_type_ids"] for e in ejemplos]).to(dispositivo),
        "attention_mask": torch.stack([e["attention_mask"] for e in ejemplos]).to(dispositivo),
        "labels": torch.stack([e["labels"] for e in ejemplos]).to(dispositivo),
    }

dataset_mlm = construir_dataset_mlm(corpus, repeticiones=25, max_len=MAX_LEN)
print("Ejemplos MLM:", len(dataset_mlm))
print("Forma de un ejemplo:", dataset_mlm[0]["input_ids"].shape)

### **Entrenamiento MLM simplificado**

#### **Interpretación de la pérdida**

La pérdida se calcula solo en posiciones enmascaradas. Las posiciones con label `-100` se ignoran. Esta convención es común en PyTorch para tareas donde no todas las posiciones deben contribuir al error.

In [ ]:
optimizador = torch.optim.AdamW(modelo.parameters(), lr=3e-3, weight_decay=0.01)
batch_size = 16
pasos = 80

modelo.train()
historial_perdida = []

for paso in range(1, pasos + 1):
    lote_indices = random.sample(range(len(dataset_mlm)), k=batch_size)
    lote = crear_lote([dataset_mlm[i] for i in lote_indices])

    logits_mlm, _ = modelo(
        lote["input_ids"],
        lote["token_type_ids"],
        lote["attention_mask"],
    )

    perdida = F.cross_entropy(
        logits_mlm.view(-1, len(vocabulario)),
        lote["labels"].view(-1),
        ignore_index=-100,
    )

    optimizador.zero_grad()
    perdida.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
    optimizador.step()

    historial_perdida.append(float(perdida.item()))

    if paso % 20 == 0:
        print(f"Paso {paso:03d} - pérdida MLM: {perdida.item():.4f}")

print("Entrenamiento MLM terminado.")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(historial_perdida)
plt.title("Pérdida MLM durante el entrenamiento")
plt.xlabel("Paso")
plt.ylabel("Pérdida")
plt.grid(True)
plt.show()

### **Inferencia MLM**

#### **Predicción de tokens ocultos**

Después de entrenar, podemos ocultar un token y observar las principales predicciones del modelo. En BERT real, esta tarea se aprende con corpus masivo. Aquí solo buscamos mostrar el flujo completo.

In [ ]:
def predecir_mascara(modelo, texto, top_k=8, max_len=MAX_LEN):
    """Predice candidatos para la posición [MASK] de una oración."""
    codificado = codificar_bert(texto, max_len=max_len)
    input_ids = codificado["input_ids"].unsqueeze(0).to(dispositivo)
    token_type_ids = codificado["token_type_ids"].unsqueeze(0).to(dispositivo)
    attention_mask = codificado["attention_mask"].unsqueeze(0).to(dispositivo)

    posiciones_mask = (input_ids[0] == MASK_ID).nonzero(as_tuple=True)[0]
    if len(posiciones_mask) == 0:
        raise ValueError("El texto debe incluir el token [MASK].")

    modelo.eval()
    with torch.no_grad():
        logits, _ = modelo(input_ids, token_type_ids, attention_mask)
        pos = posiciones_mask[0].item()
        probs = F.softmax(logits[0, pos], dim=-1)
        valores, indices = torch.topk(probs, k=top_k)

    return pd.DataFrame({
        "token_predicho": [id_a_token[i.item()] for i in indices],
        "probabilidad": [float(v.item()) for v in valores],
    })

predecir_mascara(modelo, "el transformer encoder usa [MASK] bidireccional", top_k=8)

### **Inspección de atención**

#### **Qué se debe observar**

Los pesos de atención permiten estudiar qué tokens consulta el modelo en cada capa y cabecera. No deben interpretarse como explicación completa del modelo, pero sí como una herramienta útil para analizar patrones de dependencia.

In [ ]:
texto_atencion = "bert aprende representaciones contextuales del lenguaje"
codificado = codificar_bert(texto_atencion, max_len=MAX_LEN)

input_ids = codificado["input_ids"].unsqueeze(0).to(dispositivo)
token_type_ids = codificado["token_type_ids"].unsqueeze(0).to(dispositivo)
attention_mask = codificado["attention_mask"].unsqueeze(0).to(dispositivo)

modelo.eval()
with torch.no_grad():
    logits, salida_cls, atenciones = modelo(
        input_ids,
        token_type_ids,
        attention_mask,
        devolver_atencion=True,
    )

tokens_visibles = codificado["tokens"]
pesos = atenciones[-1][0, 0].detach().cpu().numpy()
pesos_visibles = pesos[:len(tokens_visibles), :len(tokens_visibles)]

pd.DataFrame(
    np.round(pesos_visibles, 3),
    index=tokens_visibles,
    columns=tokens_visibles,
)

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(pesos_visibles)
plt.xticks(range(len(tokens_visibles)), tokens_visibles, rotation=45, ha="right")
plt.yticks(range(len(tokens_visibles)), tokens_visibles)
plt.title("Atención de una cabecera en la última capa")
plt.xlabel("Token atendido")
plt.ylabel("Token que consulta")
plt.tight_layout()
plt.show()

### **Freezing y fine-tuning**

#### **Por qué freezing importa**

Freezing significa congelar parámetros preentrenados para que no se actualicen durante una tarea nueva. Esto es importante en multimodalidad porque muchos sistemas modernos congelan componentes costosos:

| Estrategia | Qué se entrena | Ventaja | Riesgo |
|---|---|---|---|
| Freezing completo del encoder | Solo una cabecera ligera | Bajo costo | Menor adaptación |
| Fine-tuning parcial | Últimas capas y cabecera | Balance entre costo y adaptación | Requiere decisión cuidadosa |
| Fine-tuning completo | Todo el modelo | Mayor capacidad de adaptación | Mayor costo y riesgo de sobreajuste |
| Proyector entrenable | Solo puente entre modalidades | Muy usado en MLLMs | Depende de la calidad del encoder congelado |

BLIP-2 y LLaVA usan esta lógica: no siempre se entrena todo. Muchas veces se entrena un puente entre visión y lenguaje.

In [ ]:
def contar_parametros_entrenables(modelo):
    """Cuenta parámetros totales y entrenables."""
    total = sum(p.numel() for p in modelo.parameters())
    entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
    return total, entrenables

total, entrenables = contar_parametros_entrenables(modelo)
print("Antes de congelar")
print("Parámetros totales:", total)
print("Parámetros entrenables:", entrenables)

# Congelar el encoder completo de BERT mini
for parametro in modelo.parameters():
    parametro.requires_grad = False

total, entrenables = contar_parametros_entrenables(modelo)
print("\nDespués de congelar")
print("Parámetros totales:", total)
print("Parámetros entrenables:", entrenables)

### **Clasificación con encoder congelado**

#### **Demostración de transferencia**

Ahora usamos el encoder congelado como extractor de características. Entrenamos una cabecera lineal pequeña sobre el token `[CLS]`. Esto reproduce una estrategia común: mantener fijo un backbone y entrenar solo una cabecera o proyector.

In [ ]:
datos_clasificacion = [
    ("bert aprende representaciones contextuales del lenguaje", "texto"),
    ("el transformer encoder usa atencion bidireccional", "texto"),
    ("masked language modeling oculta tokens", "texto"),
    ("los embeddings de posicion conservan el orden", "texto"),
    ("visualbert fusiona regiones visuales y tokens textuales", "multimodal"),
    ("uniter aprende alineamiento entre palabras y regiones", "multimodal"),
    ("vilt procesa patches visuales con texto", "multimodal"),
    ("blip dos conecta vision y lenguaje", "multimodal"),
    ("llava usa instrucciones multimodales", "multimodal"),
    ("qwen vl analiza documentos y diagramas", "multimodal"),
    ("freezing permite entrenar pocas capas", "transferencia"),
    ("fine tuning completo actualiza parametros", "transferencia"),
    ("un proyector conecta embeddings visuales y textuales", "transferencia"),
    ("el grounding exige evidencia visual", "transferencia"),
]

etiquetas = sorted(set(etiqueta for _, etiqueta in datos_clasificacion))
etiqueta_a_id = {etiqueta: i for i, etiqueta in enumerate(etiquetas)}
id_a_etiqueta = {i: etiqueta for etiqueta, i in etiqueta_a_id.items()}

class ClasificadorCLS(nn.Module):
    """cabecera lineal sobre el token CLS."""

    def __init__(self, d_model, num_clases):
        super().__init__()
        self.cabecera = nn.Linear(d_model, num_clases)

    def forward(self, cls):
        return self.cabecera(cls)

clasificador = ClasificadorCLS(d_model=64, num_clases=len(etiquetas)).to(dispositivo)
opt_clf = torch.optim.AdamW(clasificador.parameters(), lr=5e-2)

def obtener_cls_congelado(textos):
    """Obtiene vectores CLS sin actualizar el encoder."""
    codificados = [codificar_bert(texto, max_len=MAX_LEN) for texto in textos]
    input_ids = torch.stack([c["input_ids"] for c in codificados]).to(dispositivo)
    token_type_ids = torch.stack([c["token_type_ids"] for c in codificados]).to(dispositivo)
    attention_mask = torch.stack([c["attention_mask"] for c in codificados]).to(dispositivo)

    modelo.eval()
    with torch.no_grad():
        _, cls = modelo(input_ids, token_type_ids, attention_mask)
    return cls

textos = [x for x, _ in datos_clasificacion]
y = torch.tensor([etiqueta_a_id[e] for _, e in datos_clasificacion], dtype=torch.long).to(dispositivo)

for epoca in range(1, 101):
    cls = obtener_cls_congelado(textos)
    logits = clasificador(cls)
    perdida = F.cross_entropy(logits, y)

    opt_clf.zero_grad()
    perdida.backward()
    opt_clf.step()

    if epoca % 25 == 0:
        pred = logits.argmax(dim=-1)
        exactitud = (pred == y).float().mean().item()
        print(f"Época {epoca:03d} - pérdida: {perdida.item():.4f} - exactitud: {exactitud:.3f}")

print("Entrenamiento de cabecera lineal terminado.")

In [ ]:
cls = obtener_cls_congelado(textos)
with torch.no_grad():
    logits = clasificador(cls)
    predicciones = logits.argmax(dim=-1).cpu().tolist()

pd.DataFrame({
    "texto": textos,
    "etiqueta_real": [e for _, e in datos_clasificacion],
    "prediccion": [id_a_etiqueta[i] for i in predicciones],
})

### **Atención bidireccional como base de fusión multimodal**

#### **De tokens textuales a tokens visuales**

En BERT, todos los tokens textuales comparten el mismo espacio de dimensión `d_model`. En modelos multimodales, se busca que los tokens visuales también entren a ese espacio.

La transición conceptual es:

| Sistema | Secuencia de entrada |
|---|---|
| BERT | `[CLS] texto [SEP]` |
| VisualBERT | `[CLS] texto [SEP] regiones visuales` |
| UNITER | palabras y regiones alineadas |
| ViLT | texto y patches visuales |
| BLIP-2 | queries visuales y LLM congelado |
| LLaVA | embeddings visuales proyectados hacia el LLM |

La clave es que la modalidad visual debe ser convertida a una secuencia compatible con el Transformer.

In [ ]:
class ProyectorVisualMini(nn.Module):
    """Proyecta tokens visuales sintéticos al espacio del encoder textual."""

    def __init__(self, dim_visual=128, d_model=64):
        super().__init__()
        self.proyector = nn.Sequential(
            nn.Linear(dim_visual, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tokens_visuales):
        return self.proyector(tokens_visuales)

# Crear una entrada textual real usando embeddings de BERT mini
texto = "el grounding reduce respuestas sin evidencia visual"
codificado = codificar_bert(texto, max_len=MAX_LEN)
input_ids = codificado["input_ids"].unsqueeze(0).to(dispositivo)
token_type_ids = codificado["token_type_ids"].unsqueeze(0).to(dispositivo)
attention_mask = codificado["attention_mask"].unsqueeze(0).to(dispositivo)

# Recuperar embeddings textuales antes del encoder
with torch.no_grad():
    embeddings_texto = modelo.embeddings(input_ids, token_type_ids)

# Simular 4 tokens visuales, por ejemplo patches o regiones
batch_size = 1
num_tokens_visuales = 4
dim_visual = 128
tokens_visuales_crudos = torch.randn(batch_size, num_tokens_visuales, dim_visual).to(dispositivo)

proyector_visual = ProyectorVisualMini(dim_visual=dim_visual, d_model=64).to(dispositivo)
embeddings_visuales = proyector_visual(tokens_visuales_crudos)

# Concatenar texto e imagen en una sola secuencia
embeddings_multimodales = torch.cat([embeddings_texto, embeddings_visuales], dim=1)

print("Embeddings de texto:", tuple(embeddings_texto.shape))
print("Embeddings visuales proyectados:", tuple(embeddings_visuales.shape))
print("Secuencia multimodal combinada:", tuple(embeddings_multimodales.shape))

### **Tipos de objetivos de preentrenamiento**

#### **De MLM a objetivos multimodales**

BERT usa MLM. Los modelos multimodales agregan objetivos que obligan a alinear texto e imagen.

| Objetivo | Significado | Uso típico |
|---|---|---|
| MLM | predecir tokens textuales ocultos | BERT, VisualBERT, UNITER |
| MRM | predecir regiones visuales ocultas | LXMERT, UNITER |
| ITM | decidir si imagen y texto corresponden | VisualBERT, UNITER, BLIP |
| WRA | alinear palabras y regiones | UNITER |
| Contrastive loss | acercar pares correctos y alejar incorrectos | CLIP, CoCa |
| Captioning loss | generar texto condicionado por imagen | BLIP, CoCa |
| Instruction tuning | responder instrucciones multimodales | LLaVA, Qwen2.5-VL, InternVL |

Para trabajar con Transformers multimodales, el estudiante debe reconocer qué objetivo entrena qué capacidad.

### **BERT frente a modelos multimodales**

#### **Lectura comparativa mínima**

| Modelo | Qué hereda de BERT | Qué agrega |
|---|---|---|
| VisualBERT | encoder bidireccional y `[CLS]` | regiones visuales como tokens |
| ViLBERT | atención y representación contextual | dos flujos con co-attention |
| LXMERT | encoder textual | encoder visual y encoder cross-modal |
| UNITER | single-stream bidireccional | objetivos de alineamiento multimodal |
| ViLT | fusión en una pila Transformer | patches visuales sin detector pesado |
| BLIP-2 | freezing y puente aprendible | Q-Former entre visión y LLM |
| LLaVA | freezing y proyección | instruction tuning visual |
| Qwen2.5-VL | LLM multimodal | documentos, grounding, video y agente visual |
| InternVL | escalamiento de VLMs | razonamiento, evaluación y capacidades abiertas |

El punto central es que BERT no es el destino final del curso. Es la base que permite entender cómo se fusionan, congelan, proyectan y adaptan representaciones.

### **Demostración opcional con Hugging Face**

#### **Uso con modelos reales**

La siguiente celda está desactivada por defecto. Si el entorno Docker o Colab ya tiene el modelo descargado, puede activarse cambiando `EJECUTAR_DEMO_HF` a `True`.

La opción `local_files_only=True` evita descargas automáticas y hace más reproducible la práctica en laboratorios.

In [ ]:
EJECUTAR_DEMO_HF = False

if EJECUTAR_DEMO_HF:
    from transformers import AutoTokenizer, AutoModelForMaskedLM

    nombre_modelo = "bert-base-multilingual-cased"

    tokenizer = AutoTokenizer.from_pretrained(nombre_modelo, local_files_only=True)
    modelo_hf = AutoModelForMaskedLM.from_pretrained(nombre_modelo, local_files_only=True).to(dispositivo)

    texto = "La atención bidireccional permite usar contexto izquierdo y [MASK]."
    entradas = tokenizer(texto, return_tensors="pt").to(dispositivo)

    with torch.no_grad():
        salidas = modelo_hf(**entradas)

    pos_mask = (entradas["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()
    probs = torch.softmax(salidas.logits[0, pos_mask], dim=-1)
    valores, indices = torch.topk(probs, k=10)

    for valor, indice in zip(valores, indices):
        print(tokenizer.decode([indice.item()]), float(valor.item()))
else:
    print("Demo Hugging Face desactivada. El cuaderno principal no requiere descargas externas.")

### **Actividad guiada 1**

#### **Construcción de entrada tipo BERT**

Construye una entrada con dos segmentos:

1. Segmento A: una pregunta sobre un modelo multimodal.
2. Segmento B: una respuesta breve.

Debes mostrar tokens, ids, ids de segmento y máscara de atención. Explica qué cambiaría si el segmento B fuera una imagen representada como tokens visuales.

In [ ]:
# Espacio de trabajo del estudiante
pregunta = "que hace un encoder multimodal"
respuesta = "fusiona tokens textuales y visuales"

ejercicio = codificar_bert(pregunta, respuesta, max_len=18)

pd.DataFrame({
    "posicion": list(range(18)),
    "token": ejercicio["tokens"] + ["[PAD]"] * (18 - len(ejercicio["tokens"])),
    "input_id": ejercicio["input_ids"].tolist(),
    "segmento": ejercicio["token_type_ids"].tolist(),
    "mascara": ejercicio["attention_mask"].tolist(),
})

### **Actividad guiada 2**

#### **Freezing y transferencia**

Explica por qué freezing puede ser una decisión razonable cuando se trabaja con modelos grandes. Después modifica el cuaderno para comparar:

1. Encoder congelado.
2. Última capa descongelada.
3. Todo el modelo descongelado.

Registra cantidad de parámetros entrenables, pérdida final y tiempo aproximado.

In [ ]:
# Espacio de trabajo del estudiante
def cambiar_freezing(modelo, modo="congelado"):
    """Configura qué partes del modelo se entrenan."""
    for parametro in modelo.parameters():
        parametro.requires_grad = False

    if modo == "ultima_capa":
        for parametro in modelo.capas[-1].parameters():
            parametro.requires_grad = True
    elif modo == "completo":
        for parametro in modelo.parameters():
            parametro.requires_grad = True
    elif modo == "congelado":
        pass
    else:
        raise ValueError("Modo no reconocido.")

    total, entrenables = contar_parametros_entrenables(modelo)
    return {"modo": modo, "total": total, "entrenables": entrenables}

resultados_freezing = [
    cambiar_freezing(modelo, "congelado"),
    cambiar_freezing(modelo, "ultima_capa"),
    cambiar_freezing(modelo, "completo"),
]

pd.DataFrame(resultados_freezing)

### **Actividad guiada 3**

#### Puente hacia Transformers multimodales

Completa una breve ficha técnica para cada modelo:

| Modelo | Herencia de BERT | Componente multimodal | Objetivo de entrenamiento |
|---|---|---|---|
| VisualBERT |  |  |  |
| UNITER |  |  |  |
| ViLT |  |  |  |
| BLIP-2 |  |  |  |
| LLaVA |  |  |  |

La respuesta debe justificar por qué cada modelo necesita una representación común entre texto e imagen.

### **Ejercicios**

#### **Ejercicio 1**

Implementa una variante donde el modelo use embeddings de modalidad. Use `0` para texto y `1` para tokens visuales. Compara esta idea con segment embeddings.

#### **Ejercicio 2**

Explica por qué MLM no es suficiente para entrenar un buen modelo multimodal. Proponga al menos dos objetivos adicionales y justifica qué capacidad aprende cada uno.

#### **Ejercicio 3**

Compara la atención bidireccional, co-attention y cross-attention. Usa como referencia conceptual BERT, ViLBERT, LXMERT y BLIP-2.

#### **Ejercicio 4**

Entrena la cabecera de clasificación con encoder congelado y luego con la última capa descongelada. Compara la pérdida, exactitud y número de parámetros entrenables.

#### **Ejercicio 5**

Diseña una secuencia multimodal con tokens textuales, tokens visuales y un token global `[CLS]`. Describe qué partes corresponderían a VisualBERT, ViLT y LLaVA.

In [ ]:
## Tus respuestas

### Bibliografía mínima sugerida

#### Lecturas para conectar con el siguiente cuaderno

| Tema | Lectura recomendada |
|---|---|
| BERT | Devlin et al., BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding |
| Transformer | Vaswani et al., Attention Is All You Need |
| VisualBERT | Li et al., VisualBERT: A Simple and Performant Baseline for Vision and Language |
| UNITER | Chen et al., UNITER: Universal Image-Text Representation Learning |
| ViLT | Kim et al., ViLT: Vision-and-Language Transformer Without Convolution or Region Supervision |
| BLIP-2 | Li et al., BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and Large Language Models |
| LLaVA | Liu et al., Visual Instruction Tuning |